In [1]:
from pyspark.sql import SparkSession

def calculate_red_violations(data_source, output_uri):
    """Procesa las inspecciones, obtiene el Top 10 de establecimientos con más violaciones RED y guarda el resultado en S3."""

    # Crear o reutilizar una sesión de Spark
    spark = SparkSession.builder.appName("Food Establishment - RED Violations").getOrCreate()

    # Leer el archivo CSV desde S3 y crear un DataFrame
    restaurants_df = spark.read.option("header", "true").option("inferSchema", "true").csv(data_source)

    # Crear una vista temporal para realizar consultas con Spark SQL
    restaurants_df.createOrReplaceTempView("restaurant_violations")

    # Filtrar las violaciones RED, agrupar por establecimiento y obtener el Top 10
    result_df = spark.sql("""
        SELECT name, COUNT(*) AS total_red_violations
        FROM restaurant_violations
        WHERE violation_type = 'RED'
        GROUP BY name
        ORDER BY total_red_violations DESC
        LIMIT 10
    """)

    # Mostrar el resultado
    print("Top 10 establecimientos con más violaciones RED:")
    result_df.show(truncate=False)

    # Guardar el resultado en S3 en formato CSV
    result_df.write.option("header", "true").mode("overwrite").csv(output_uri)

    # Retornar el DataFrame con los resultados
    return result_df

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1790401439146_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
# Definir las rutas de entrada y salida en Amazon S3
data_source = "s3://seccion-b-rosa-lab1/food_establishment_data.csv"
output_uri = "s3://seccion-b-rosa-lab1/output/red_violations/"

# Ejecutar la función
result_df = calculate_red_violations(data_source, output_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Top 10 establecimientos con m?s violaciones RED:
+----------------------+--------------------+
|name                  |total_red_violations|
+----------------------+--------------------+
|SUBWAY                |322                 |
|T-MOBILE PARK         |315                 |
|WHOLE FOODS MARKET    |299                 |
|PCC COMMUNITY MARKETS |251                 |
|TACO TIME             |240                 |
|MCDONALD'S            |177                 |
|THAI GINGER           |153                 |
|SAFEWAY INC #1508     |143                 |
|TAQUERIA EL RINCONSITO|134                 |
|HIMITSU TERIYAKI      |128                 |
+----------------------+--------------------+